In [1]:
# Libraries
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="0"
from kerasbackend import kerasbackend
KERASBACKEND = kerasbackend('tensorflow')
###############################################################################
import os
import json
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Lambda
from keras.layers import Conv2D, MaxPooling2D, Cropping2D
from keras.callbacks import ModelCheckpoint
from keras.optimizers import SGD
from keras.optimizers import RMSprop, Adam
from keras.layers.advanced_activations import LeakyReLU, PReLU
from keras.layers import Input, Dropout, Flatten, MaxPooling2D, Dense, Activation
import tensorflow as tf
from keras import backend as K
from keras.models import model_from_json
from keras.callbacks import ModelCheckpoint, Callback, EarlyStopping
import pandas as pd
import numpy as np
import cv2
import matplotlib.pylab as plt
import random
from params import params
from keras.models import Model
from keras.layers import Input
from keras.layers import Dense
from keras.layers import Flatten
from keras.layers import Dropout
from keras.layers.convolutional import Conv2D
from keras.layers.pooling import MaxPooling2D
from keras.layers.merge import concatenate

Keras backend is:  tensorflow


Using TensorFlow backend.
/home/ikaya/anaconda3/envs/py36/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:526: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
/home/ikaya/anaconda3/envs/py36/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:527: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
/home/ikaya/anaconda3/envs/py36/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:528: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int16, 1)])
/home/ikaya/anaconda3/envs/py

In [2]:
###################### Tensorflow Ram Usage ####################################################################

if KERASBACKEND.KERAS_BACKEND == 'tensorflow':
    # TensorFlow wizardry
    config = tf.ConfigProto() 
    # Don't pre-allocate memory; allocate as-needed
    config.gpu_options.allow_growth = True 
    # Only allow a total of half the GPU memory to be allocated
    config.gpu_options.per_process_gpu_memory_fraction = 1.0 
    # Create a session with the above options specified.
    K.tensorflow_backend.set_session(tf.Session(config=config))

##################################################################################################################################


In [3]:
# PreProcess
rOWS     = 128
cOLS     = 128
cHANNELS = 1

### Read the Training Data ###
TRAIN_DIR    = '../Data/BelgiumTrafficSign/Training/'

directories  = [TRAIN_DIR+i for i in os.listdir(TRAIN_DIR)] # use this for full dataset
address      = []
for ii in os.listdir(TRAIN_DIR):
    address.append(ii)

label            = []
label_enc        = []
train_images_all = []
for cntr in range(len(directories)):
    label_arr = np.zeros((1,len(directories)), dtype='uint8')
    label_arr[0,int(address[cntr])] = 1
    label_list = []
    for i in range(len(directories)):
        label_list.append(0)
    label_list[int(address[cntr])] = 1
    for ii in os.listdir(directories[cntr]+'/'):
        if '.ppm' in ii:
            train_images_all.append(directories[cntr]+'/'+ii)
            label.append(int(address[cntr]))
            label_enc.append(label_list)
    cntr = cntr + 1

train_images = []
for i in range(len(directories)):
    train_images.append(None)

for ii in range(len(directories)):
    filess = [directories[ii]+'/'+i for i in os.listdir(directories[ii]+'/') if '.ppm' in i]
    train_images[int(address[ii])] = filess

total_file_number = 0
for ii in range(len(train_images)):
    total_file_number = total_file_number + len(train_images[ii])
####################################

In [4]:
### Read the Test Data ###
TEST_DIR  = '../Data/BelgiumTrafficSign/Testing/'

directories_test  = [TEST_DIR+i for i in os.listdir(TEST_DIR)] # use this for full dataset
address_test      = []
for ii in os.listdir(TEST_DIR):
    address_test.append(ii)

label_test      = []
label_enc_test  = []
test_images_all = []
for cntr in range(len(directories_test)):
    label_arr_test = np.zeros((1,len(directories_test)), dtype='uint8')
    label_arr_test[0,int(address_test[cntr])] = 1
    label_list = []
    for i in range(len(directories)):
        label_list.append(0)
    label_list[int(address[cntr])] = 1
    for ii in os.listdir(directories_test[cntr]+'/'):
        if '.ppm' in ii:
            test_images_all.append(directories_test[cntr]+'/'+ii)
            label_test.append(int(address_test[cntr]))
            label_enc_test.append(label_list)
    cntr = cntr + 1

test_images = []
for i in range(len(directories_test)):
    test_images.append('None')

for ii in range(len(directories_test)):
    filess_test = [directories_test[ii]+'/'+i for i in os.listdir(directories_test[ii]+'/') if '.ppm' in i]
    test_images[int(address_test[ii])] = filess_test

total_file_number_test = 0
for ii in range(len(test_images)):
    total_file_number_test = total_file_number_test + len(test_images[ii])
###################################    with mx.gpu(device_id=0):
#        with mx.gpu(device_id=1):
#     with tf.device('/gpu:0'):## 

### Shuffle ###
reindex = list(range(len(train_images_all)))
random.seed(random.randint(0,1000))
random.shuffle(reindex)

reindex_test = list(range(len(test_images_all)))
random.seed(random.randint(0,1000))
random.shuffle(reindex_test)
###############

### Read Image ###
def read_image(file_path):
    img = cv2.imread(file_path, cv2.IMREAD_GRAYSCALE) #cv2.IMREAD_GRAYSCALE
    return cv2.resize(img, (rOWS, cOLS), interpolation=cv2.INTER_AREA)

def prep_data(images):
    count = len(images)
    data = np.ndarray((count, rOWS, cOLS, cHANNELS), dtype=np.uint8)

    for i, image_file in enumerate(images):
        image = read_image(image_file)
        data[i,:,:,0] = image
        if i%100 == 0: print('Processed {} of {}'.format(i, count))
    
    return data
#####################

# Prepare the input and output ###
train_input         = prep_data(train_images_all)
train_input_shuffle = np.ndarray((len(train_images_all),rOWS,cOLS,cHANNELS),dtype='uint8')
train_label_shuffle = []
for ii in range(len(train_images_all)):
    train_input_shuffle[ii,:,:,:] = train_input[reindex[ii],:,:,:]
    train_label_shuffle.append(label_enc[reindex[ii]])
train_label_shuffle_arr = np.array(train_label_shuffle)

test_input         = prep_data(test_images_all)
test_input_shuffle = np.ndarray((len(test_images_all),rOWS,cOLS,cHANNELS),dtype='uint8')
test_label_shuffle = []
for ii in range(len(test_images_all)):
    test_input_shuffle[ii,:,:,:] = test_input[reindex_test[ii],:,:,:]
    test_label_shuffle.append(label_enc_test[reindex_test[ii]])
test_label_shuffle_arr = np.array(test_label_shuffle)

print("Train shape: {}".format(train_input_shuffle.shape))
print("Test shape: {}".format(test_input_shuffle.shape))
####################################

### Show the Data ###
def show_data():
    for i in range(10):
        ix = np.random.randint(0,len(train_images_all))
        plt.imshow(train_input_shuffle[ix,:,:,0],cmap='gray')
        plt.show()
        print('label = ', np.argmax(train_label_shuffle[ix][0]))
####################################

Processed 0 of 4575
Processed 100 of 4575
Processed 200 of 4575
Processed 300 of 4575
Processed 400 of 4575
Processed 500 of 4575
Processed 600 of 4575
Processed 700 of 4575
Processed 800 of 4575
Processed 900 of 4575
Processed 1000 of 4575
Processed 1100 of 4575
Processed 1200 of 4575
Processed 1300 of 4575
Processed 1400 of 4575
Processed 1500 of 4575
Processed 1600 of 4575
Processed 1700 of 4575
Processed 1800 of 4575
Processed 1900 of 4575
Processed 2000 of 4575
Processed 2100 of 4575
Processed 2200 of 4575
Processed 2300 of 4575
Processed 2400 of 4575
Processed 2500 of 4575
Processed 2600 of 4575
Processed 2700 of 4575
Processed 2800 of 4575
Processed 2900 of 4575
Processed 3000 of 4575
Processed 3100 of 4575
Processed 3200 of 4575
Processed 3300 of 4575
Processed 3400 of 4575
Processed 3500 of 4575
Processed 3600 of 4575
Processed 3700 of 4575
Processed 3800 of 4575
Processed 3900 of 4575
Processed 4000 of 4575
Processed 4100 of 4575
Processed 4200 of 4575
Processed 4300 of 4575


In [5]:
### Create the CNN Model ###
def create_model(x1,x2,do):
    
    optimizer = Adam(lr=0.0001, beta_1=0.9, beta_2=0.999, epsilon=1e-08, decay=0.0)
    
    inp = Input(shape=(rOWS, cOLS, cHANNELS),name='input_cnn')
    
    Layer = inp
    
    for ii in x1:
        Layer = Conv2D(ii, (3,3), padding='same', activation='relu')(Layer)
        Layer = Conv2D(ii, (3,3), padding='same', activation='relu')(Layer)
        Layer = MaxPooling2D(pool_size=(2,2))(Layer)
    
    Layer = Flatten()(Layer)
    
    for ii in x2:
        Layer = Dense(ii, activation='relu', kernel_initializer='he_normal')(Layer)
        Layer = Dropout(do)(Layer)
    
    out = Dense(62, activation='softmax', kernel_initializer='he_normal')(Layer)
    
    model = Model(inputs= inp, outputs= out)
    
    model.compile(loss='binary_crossentropy',optimizer=optimizer)
    model.summary()
    return model

In [22]:
# Loss History and Model
class LossHistory(Callback):
    def on_train_begin(self, logs={}):
        self.losses = []
        self.val_losses = []
        
    def on_epoch_end(self, batch, logs={}):
        self.losses.append(logs.get('loss'))
        self.val_losses.append(logs.get('val_loss'))
        
### RUN the Model ###

parameters = {}
x1 = [[20,40,20],[50,100,100,50],[50,100,150,100,50],[50,150,200,240,190,400]]
x2 = [[50,100],[100,100],[100,150,100],[400,300,80]]
do = [0.1,0.3,0.3,0.5]

def run_model(x1,x2,do):
    model           = create_model(x1,x2,do)
    early_stopping  = EarlyStopping(monitor='val_loss', patience=5, verbose=1, mode='auto') 
    history         = LossHistory() 
    model.fit(np.array(train_input_shuffle), np.array(train_label_shuffle), batch_size=64, nb_epoch=2, validation_split=0.10, verbose=1, shuffle=True, callbacks=[history, early_stopping])
    return history, model

In [23]:
# Run
for ii in range(len(x1)):
    history, model = run_model(x1[ii],x2[ii],do[ii])
    parameters['model_name']  = "model"+str(ii+1)
    parameters['conv_layer']  = x1[ii]
    parameters['flat_layer']  = x2[ii]
    parameters['dropout']     = do[ii]
    parameters['val_loss']    = history.val_losses[-1]
    parameters['# of params'] = model.count_params()
    params(parameters=parameters,filename='params.txt')
    

_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_cnn (InputLayer)       (None, 128, 128, 1)       0         
_________________________________________________________________
conv2d_55 (Conv2D)           (None, 128, 128, 20)      200       
_________________________________________________________________
conv2d_56 (Conv2D)           (None, 128, 128, 20)      3620      
_________________________________________________________________
max_pooling2d_28 (MaxPooling (None, 64, 64, 20)        0         
_________________________________________________________________
conv2d_57 (Conv2D)           (None, 64, 64, 40)        7240      
_________________________________________________________________
conv2d_58 (Conv2D)           (None, 64, 64, 40)        14440     
_________________________________________________________________
max_pooling2d_29 (MaxPooling (None, 32, 32, 40)        0         
__________

/home/ikaya/anaconda3/envs/py36/lib/python3.6/site-packages/ipykernel_launcher.py:21: UserWarning: The `nb_epoch` argument in `fit` has been renamed `epochs`.


Train on 4117 samples, validate on 458 samples
Epoch 1/2
4117/4117 [==============================] - 4s 1ms/step - loss: 0.1124 - val_loss: 0.0780
Epoch 2/2
4117/4117 [==============================] - 3s 673us/step - loss: 0.0757 - val_loss: 0.0701

file params.txt is created

_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_cnn (InputLayer)       (None, 128, 128, 1)       0         
_________________________________________________________________
conv2d_61 (Conv2D)           (None, 128, 128, 50)      500       
_________________________________________________________________
conv2d_62 (Conv2D)           (None, 128, 128, 50)      22550     
_________________________________________________________________
max_pooling2d_31 (MaxPooling (None, 64, 64, 50)        0         
_________________________________________________________________
conv2d_63 (Conv2D)           (None, 64, 64, 100)       45100

Train on 4117 samples, validate on 458 samples
Epoch 1/2
4117/4117 [==============================] - 16s 4ms/step - loss: 0.0823 - val_loss: 0.0804
Epoch 2/2
4117/4117 [==============================] - 12s 3ms/step - loss: 0.0799 - val_loss: 0.0737

file exists



In [ ]:
####################### POST PROCESSING ##########################################################
### Predict / Show / Save ###
predictions     = model.predict(train_input_shuffle)
prediction_test = model.predict(test_input_shuffle)
list_wrong_prediction = []
def show(input,label,predictions,adet):
    cnt = 0
    list_wrong_prediction = []
    for i in range(adet):
        plt.imshow(input[i,:,:,0],cmap='gray')
        plt.show()        
        if np.argmax(predictions[i]) == np.argmax(label[i]):
            cnt = cnt + 1
        else:
            wrong_prediction = {'prediction':np.argmax(predictions[i]), 'label':np.argmax(label[i])}
            list_wrong_prediction.append(wrong_prediction)
        accuracy = 100 * cnt / (i+1)
        print('image no = ', i+1)
        print('prediction = ', np.argmax(predictions[i]))
        print('label = ', np.argmax(label[i]))
        print('accuracy = ', accuracy)
        print('wrong prediction no = ', i+1-cnt)
    return list_wrong_prediction

In [ ]:
# find the wrong predicted ones in training set
list_wrong_prediction = show(train_input_shuffle,train_label_shuffle,predictions,len(train_images_all))

In [ ]:
# find the wrong predicted ones in test set
list_wrong_prediction = show(test_input_shuffle,test_label_shuffle,prediction_test,len(test_images_all))

In [ ]:
# list the wrong prediction labels
list_wrong_prediction_labels = np.zeros((len(label_list),1))
for ii in range(len(list_wrong_prediction)):
    list_wrong_prediction_labels[list_wrong_prediction[ii]['label'],0] = list_wrong_prediction_labels[list_wrong_prediction[ii]['label'],0] + 1
    
np.argmax(list_wrong_prediction_labels)

In [ ]:
### save the model
def save(model_name,model):
    model_json = model.to_json()
    with open(model_name + ".json","w") as json_file:
        json_file.write(model_json)
        model.save_weights(model_name + ".h5")
save('belgium_traffic_sign_01',model)
#############################